# Sweep unlinked open packages → mark `status='orphaned'`

**BUG-049 operator hotfix.**

## Problem

The BUG-046 gate (`_has_open_package_for_strategy` in `src/runtime/pipeline.py`)
queries `order_packages WHERE strategy_name=? AND status='open'` to decide
whether a strategy is allowed to generate a new signal. Any row with
`status='open'` blocks that strategy — even rows that were **never executed**
(no `linked_trade_id`, meaning no trade was ever placed at the broker).

These unlinked packages accumulate when the risk manager rejects a package
or the dispatch pipeline fails before placing an order. They are not swept
by the reconciler (which only looks at the `trades` table) and therefore
silently block the strategy forever.

**Symptom:** `/packages` shows N open packages with no linked trades;
`/signals` shows no new signals since the oldest such package was created.

## What this notebook does

1. SSHes to the VM.
2. **Previews** all `order_packages` rows with `status='open' AND linked_trade_id IS NULL`,
   grouped by strategy.
3. Pauses for you to set `CONFIRM = True` in Cell 4.
4. **Updates** matching rows to `status='orphaned'`, unblocking each strategy's
   BUG-046 gate immediately on the next tick.

## Reversibility

```sql
UPDATE order_packages SET status = 'open' WHERE status = 'orphaned'
  AND json_extract(meta, '$.orphaned_by') = 'sweep_unlinked_packages.ipynb';
```

## Required Colab Secrets

| Name | What it holds |
|---|---|
| `VM_SSH_HOST` | VM hostname or public IP |
| `VM_SSH_USER` | SSH user on the VM (usually `ubuntu`) |

In [ ]:
# Cell 1: SSH setup (mirrors rotate_api_keys.ipynb pattern).
import os, shutil, stat, subprocess, tempfile
from google.colab import drive, userdata

print('⏳ Mounting Google Drive…')
try:
    drive.mount('/content/drive')
except Exception as exc:
    print(f'⚠️  drive.mount() raised: {exc}')

DEFAULT_SSH_KEY_NAMES = [
    'ict-bot-ovm-private.key',
    'vm_ssh_key', 'id_rsa', 'id_ed25519', 'id_ecdsa',
]
DRIVE_FOLDER = '/content/drive/MyDrive/ICT_Bot_Secrets'

ssh_key_path = None
if os.path.exists('/content/drive/MyDrive'):
    for name in DEFAULT_SSH_KEY_NAMES:
        path = os.path.join(DRIVE_FOLDER, name)
        if os.path.exists(path):
            ssh_key_path = path
            break
if ssh_key_path is None:
    raise SystemExit(
        f'No SSH key found in {DRIVE_FOLDER}/. Place '
        '`ict-bot-ovm-private.key` there and re-run.'
    )
print(f'✅ SSH key located: {ssh_key_path}')

host = userdata.get('VM_SSH_HOST')
user = userdata.get('VM_SSH_USER')
if not host or not user:
    raise SystemExit('VM_SSH_HOST + VM_SSH_USER must be set in Colab Secrets.')

tmpdir = tempfile.mkdtemp(prefix='sweep-unlinked-')
key_path = os.path.join(tmpdir, 'vm_key')
shutil.copy(ssh_key_path, key_path)
os.chmod(key_path, stat.S_IRUSR | stat.S_IWUSR)

ssh_target = f'{user}@{host}'
ssh_opts = [
    '-i', key_path,
    '-o', 'StrictHostKeyChecking=accept-new',
    '-o', 'UserKnownHostsFile=' + os.path.join(tmpdir, 'known_hosts'),
    '-o', 'ConnectTimeout=15',
    '-o', 'BatchMode=yes',
]

def run_ssh(cmd, *, label, allow_fail=False):
    proc = subprocess.run(
        ['ssh'] + ssh_opts + [ssh_target, cmd],
        capture_output=True, text=True,
    )
    if proc.returncode != 0 and not allow_fail:
        print(f'❌ {label} failed (exit {proc.returncode})')
        print((proc.stderr or proc.stdout)[:500])
        raise SystemExit(f'{label} failed')
    return (proc.stdout or '').strip()

print(f'⏳ Connecting to {ssh_target} …')
run_ssh('echo connected', label='SSH connectivity')
print('✅ SSH OK')

In [ ]:
# Cell 2: PREVIEW (read-only).
# Shows every order_package row that WOULD be orphaned by Cell 4.
# Inspect before running Cell 4.

DB_PATH = f'/home/{user}/ict-trading-bot/trade_journal.db'

# Per-strategy summary.
summary_sql = (
    "SELECT strategy_name, COUNT(*) AS stuck_count, "
    "MIN(created_at) AS oldest, MAX(created_at) AS newest "
    "FROM order_packages "
    "WHERE status = 'open' AND linked_trade_id IS NULL "
    "GROUP BY strategy_name "
    "ORDER BY stuck_count DESC;"
)
summary = run_ssh(
    f'sqlite3 -header -column {DB_PATH} "{summary_sql}"',
    label='strategy summary',
)
print('Unlinked open packages by strategy (would be orphaned):')
print()
print(summary or '(none — nothing to clean up)')

# Full row listing.
detail_sql = (
    "SELECT order_package_id, strategy_name, symbol, direction, "
    "entry, sl, tp, confidence, status, created_at, updated_at "
    "FROM order_packages "
    "WHERE status = 'open' AND linked_trade_id IS NULL "
    "ORDER BY strategy_name, datetime(created_at) DESC;"
)
detail = run_ssh(
    f'sqlite3 -header -column {DB_PATH} "{detail_sql}"',
    label='detail rows',
)
print()
print('Full row detail:')
print(detail or '(no rows)')

total = run_ssh(
    f"sqlite3 {DB_PATH} \"SELECT COUNT(*) FROM order_packages "
    f"WHERE status = 'open' AND linked_trade_id IS NULL;\"",
    label='total count',
)
print(f'\nTotal rows that would change: {total}')

---

## ⚠️ Inspect Cell 2's output above

If the rows are unlinked packages you want to retire (no `linked_trade_id` means
no trade was ever placed at the broker for them), set `CONFIRM = True` in Cell 4
and run it.

After this runs, each affected strategy's BUG-046 gate will allow new signals
on the **next tick** — no trader restart required.

In [ ]:
# Cell 4: COMMIT. Set CONFIRM = True after inspecting Cell 2.
CONFIRM = False  # <- set to True to commit

if not CONFIRM:
    print('CONFIRM is False — no changes made.')
    print('Set CONFIRM = True and re-run this cell to commit.')
else:
    update_sql = (
        "UPDATE order_packages "
        "SET status = 'orphaned', "
        "updated_at = strftime('%Y-%m-%dT%H:%M:%SZ', 'now'), "
        "meta = json_set(COALESCE(meta, '{}'), "
        "'$.orphaned_at', strftime('%Y-%m-%dT%H:%M:%SZ', 'now'), "
        "'$.orphaned_by', 'sweep_unlinked_packages.ipynb', "
        "'$.orphaned_reason', 'BUG-049 — no linked_trade_id; package was never executed') "
        "WHERE status = 'open' AND linked_trade_id IS NULL;"
    )
    run_ssh(
        f'sqlite3 {DB_PATH} "{update_sql}"',
        label='UPDATE orphaned packages',
    )
    affected = run_ssh(
        f"sqlite3 {DB_PATH} \"SELECT changes();\"",
        label='row count',
    )
    print(f'✅ UPDATE complete. {affected} row(s) changed to status=\"orphaned\".')
    print()
    print('Verify on Telegram (no trader restart needed):')
    print('  /packages  → those rows should no longer appear')
    print('  /signals   → new vwap signals should resume within one tick')
    print()
    print('Rollback if needed:')
    print("  sqlite3 trade_journal.db \"UPDATE order_packages SET status='open'")
    print("    WHERE status='orphaned'")
    print("    AND json_extract(meta, '$.orphaned_by')='sweep_unlinked_packages.ipynb';\"")
    shutil.rmtree(tmpdir, ignore_errors=True)